# 03 — Regex Mastery for Resumes

**Goal:** Master regular expressions for extracting resume information.

By the end you'll extract:
- ✅ Email addresses and phone numbers
- ✅ URLs (LinkedIn, GitHub, portfolio)
- ✅ Dates and durations
- ✅ Skill names and education details
- ✅ Location information

Regex is the **workhorse** of resume parsing — fast, deterministic, and explainable.

This chapter turns raw resume text into structured fields using regular expressions — the fastest, most deterministic, and most explainable extraction tool in the NLP toolbox. You build patterns piece by piece, then assemble them into a complete resume extractor.

**Why it matters for resumes / ATS:** ATS keyword matching, contact parsing, and date normalization are regex problems. Regex is also the tool interviewers expect you to reach for first when asked to parse text — and the `re` skills here are the foundation the statistical chapters (Ch. 04+) build on.

## 1. The `re` Module — Basics

`re.search` finds the *first* match anywhere in a string and returns a `Match` object; `re.findall` returns every non-overlapping match as a list. The `Match` object carries the matched text (`.group()`) and its position (`.span()`).

**What the code does:**
- Searches the email pattern in a contact line: the stored output shows `john.doe@email.com` at span `(14, 32)` — character offsets into the original string, useful when you need to locate a field in raw text.
- `re.findall` with the same pattern confirms one email on the line.

**Try it:** give the text two emails and rerun — `search` still returns only the first, `findall` returns both. That difference decides which function each extractor uses.

In [4]:
import re

text = "Contact me at john.doe@email.com or call +1-555-123-4567"

# re.search() — find first match
match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
if match:
    print(f"Found: {match.group()}")
    print(f"Span: {match.span()}")

# re.findall() — find all matches
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
print(f"All emails: {emails}")

Found: john.doe@email.com
Span: (14, 32)
All emails: ['john.doe@email.com']


## 2. Raw Strings & Special Characters

Inside `r"..."` backslashes are literal, so `r"\d"` is backslash-plus-d — the two characters the regex engine reads as "digit". Without the `r`, Python interprets escapes first: `"\n"` becomes a newline character, and your pattern silently changes meaning.

**What the code does:**
- `repr()` of normal vs raw: `'\n'` (one newline character) vs `'\\n'` (backslash + n) — the stored output shows the pair.
- Metacharacters (`. ^ $ * + ? { } [ ] \ | ( )`) carry special meaning unless escaped; `re.search(r"\(3\.11\)", text)` matches the literal `(3.11)` in "Python (3.11) is [great]".

**Try it:** drop the backslashes (`r"(3.11)"`) and the parentheses become a *capture group* — different result, same characters.

In [7]:
# Raw strings (r"...") prevent backslash interpretation
normal = "\n"     # newline character
raw = r"\n"       # literal backslash + n

print(f"Normal: {repr(normal)}")
print(f"Raw: {repr(raw)}")

# Common metacharacters: . ^ $ * + ? { } [ ] \ | ( )
# To match literally, escape with backslash
text = "Python (3.11) is [great]"
match = re.search(r"\(3\.11\)", text)
print(f"Escaped match: {match.group() if match else 'None'}")

Normal: '\n'
Raw: '\\n'
Escaped match: (3.11)


## 3. Character Classes

`[...]` matches exactly one character from the set; `[a-z]` is a range; `[^...]` matches anything *not* in the set. Shorthand classes `\d` (digit), `\w` (word char), `\s` (whitespace) and their uppercase negations (`\D`, `\W`, `\S`) cover the common cases.

**What the code does:**
- `[Pp]ython` on "Python, pytHon, pYTHON" matches only `Python` — the stored output's one-item list shows case-sensitivity in action (`pytHon` fails on the lowercase `t`).
- `[0-9]+` pulls every digit run from "Room 42, Floor 7, Building 101": `['42', '7', '101']`.
- `[\d()]+` on the phone line groups digits and parentheses: `['(555)', '123', '4567']` — the raw material for the phone normalizer in section 7.

**Try it:** extend `[0-9]` to `[0-9a-f]` and it starts matching hex digits — classes are just sets.

In [10]:
# Character sets [...]
text = "Python, pytHon, pYTHON"

# Match 'P' or 'p' followed by 'ython'
matches = re.findall(r"[Pp]ython", text)
print(f"Case insensitive first letter: {matches}")

# Ranges
text2 = "Room 42, Floor 7, Building 101"
digits = re.findall(r"[0-9]+", text2)
print(f"Digits: {digits}")

# Shorthand classes
# \d = digit,  \w = word char,  \s = whitespace
# \D = not digit, \W = not word, \S = not whitespace
text3 = "Contact: (555) 123-4567"
phone = re.findall(r"[\d()]+", text3)
print(f"Phone parts: {phone}")

Case insensitive first letter: ['Python']
Digits: ['42', '7', '101']
Phone parts: ['(555)', '123', '4567']


## 4. Quantifiers — How Many?

Quantifiers attach to the preceding token: `*` (0 or more), `+` (1 or more), `?` (0 or 1), plus `{n}`, `{n,}`, `{n,m}`. Greedy quantifiers consume as much as possible; a trailing `?` makes them lazy (`.*?` stops at the first opportunity).

**What the code does:**
- On "Python is great!!! Really?? Yes." both `Python.*great` and `Python.*?great` return `'Python is great'` — on this short string greedy and lazy land on the same span; the difference shows when a string has *multiple* "great" occurrences.
- The email pattern `[\w.+-]+@[\w-]+\.[\w.]+` uses `+` per component; the stored output parses all three addresses, including the 17-character domain `long-domain-name.com`.

**Try it:** add a second "great" to the text — greedy spans both, lazy stops at the first. That distinction is the classic regex interview question.

In [11]:
# *    0 or more
# +    1 or more
# ?    0 or 1
# {n}  exactly n
# {n,} n or more
# {n,m} between n and m

text = "Python is great!!! Really?? Yes."

# Greedy vs lazy quantifiers
greedy = re.findall(r"Python.*great", text)
lazy = re.findall(r"Python.*?great", text)
print(f"Greedy: {greedy}")
print(f"Lazy:   {lazy}")

# Email with proper quantifiers
email_pattern = r"[\w.+-]+@[\w-]+\.[\w.]+"
text2 = "Emails: a@b.co, test@long-domain-name.com, x@y.z"
print(f"Emails: {re.findall(email_pattern, text2)}")

Greedy: ['Python is great']
Lazy:   ['Python is great']
Emails: ['a@b.co', 'test@long-domain-name.com', 'x@y.z']


## 5. Groups — Extract Specific Parts

Parentheses do two jobs: *grouping* (apply a quantifier to a unit) and *capturing* (keep the matched substring for later). Named groups `(?P<name>...)` make captures self-documenting; `(?:...)` groups without capturing.

**What the code does:**
- `Name: (?P<name>[^,]+), Email: (?P<email>[\w.@]+)` captures two named fields; `match.group('name')` / `match.group('email')` return `Srivatsa Gorti` and `srivatsa@email.com` per the stored output.
- `(?:Name|Email): ([^, ]+)` alternates the labels but captures only the value — `findall` returns `['Srivatsa', 'srivatsa@email.com']`.

**Why it matters:** named groups are how production extractors label fields — the output of this cell is already a mini structured record.

In [12]:
# Parentheses create capture groups
text = "Name: Srivatsa Gorti, Email: srivatsa@email.com"

# Named groups (?P<name>...)
pattern = r"Name: (?P<name>[^,]+), Email: (?P<email>[\w.@]+)"
match = re.search(pattern, text)
if match:
    print(f"Name:  {match.group('name')}")
    print(f"Email: {match.group('email')}")

# Non-capturing groups (?:...) — group without capturing
pattern2 = r"(?:Name|Email): ([^, ]+)"
print(f"All values: {re.findall(pattern2, text)}")

Name:  Srivatsa Gorti
Email: srivatsa@email.com
All values: ['Srivatsa', 'srivatsa@email.com']


## 6. Email Extraction — Production-Grade

The Ch. 01 email pattern was a teaching tool; this one is production-grade: `[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}`. The `{2,}` on the top-level domain rejects one-letter TLDs like `x@y.z` — which the looser section-4 pattern happily matched.

**What the code does:**
- Runs against a multi-line contact block and finds three real addresses; the multi-part TLD in `john.doe@company.co.uk` is matched whole.
- `list(set(emails))` deduplicates — the stored `Unique:` line shows the same three in different order, because set ordering is arbitrary.

**Try it:** swap this pattern for the section-1 pattern and feed it `x@y.z` — the `{2,}` TLD requirement is the difference between "works" and "production".

In [13]:
# Multiple email patterns found in resumes
text = """
Email: john.doe@company.co.uk
Alternate: johnny99@webmail.org
LinkedIn: https://linkedin.com/in/johndoe
Contact: john@gmail.com
"""

email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
emails = re.findall(email_pattern, text)
print(f"Emails found: {emails}")

# Deduplicate
unique_emails = list(set(emails))
print(f"Unique: {unique_emails}")

Emails found: ['john.doe@company.co.uk', 'johnny99@webmail.org', 'john@gmail.com']
Unique: ['johnny99@webmail.org', 'john@gmail.com', 'john.doe@company.co.uk']


## 7. Phone Number Extraction

Phone formats are chaos: `+1 (555) 123-4567`, `555-123-4567`, `+91 98765 43210`, `1.800.555.1234`. The tolerant pattern `[+]?[\d\s()-]{7,}[\d]` accepts digits, spaces, parentheses, and hyphens — at least 7 of them, then one final digit. Normalization then strips everything except digits and a leading `+`.

**What the code does:**
- Extracts three of the four lines — **`1.800.555.1234` is skipped** because `.` is not in the pattern's character class. A deliberate limitation worth remembering.
- `normalize_phone` applies `re.sub(r'[^\d+]', '', phone)` — keep digits and `+`, drop the rest; the stored output shows the normalized digit strings.

**Why it matters:** contact fields must be normalized before matching — the same phone formatted four ways is one phone, and ATS comparison needs them comparable.

In [14]:
phone_text = """
Contact: +1 (555) 123-4567
Mobile: 555-123-4567
Work: +91 98765 43210
Fax: 1.800.555.1234
"""

# Pattern for international phone numbers
phone_pattern = r"[+]?[\d\s()-]{7,}[\d]"
phones = re.findall(phone_pattern, phone_text)
for p in phones:
    print(f"Phone: {p.strip()}")

# Clean + normalize
def normalize_phone(phone: str) -> str:
    digits = re.sub(r"[^\d+]", "", phone)
    return digits

for p in phones:
    print(f"Normalized: {normalize_phone(p)}")

Phone: +1 (555) 123-4567
Phone: 555-123-4567
Phone: +91 98765 43210
Normalized: +15551234567
Normalized: 5551234567
Normalized: +919876543210


## 8. URL Extraction (LinkedIn, GitHub)

`https?://` matches both `http://` and `https://`; `[\w./-]+` then consumes the rest of the URL. Links are high-value resume signals — LinkedIn, GitHub, portfolio — and easy to categorize by substring.

**What the code does:**
- Finds three URLs from the block — **`www.buildsrivatsa.qzz.io` is missed** because it has no scheme; a `www\.` alternative in the pattern would catch it.
- Categorization is a plain `if "linkedin" in url.lower()` chain; the stored output labels the LinkedIn and GitHub links and leaves the portfolio uncategorized.

**Try it:** extend the pattern with `|www\.` and the fourth link appears — small pattern changes, big recall differences.

In [17]:
url_text = """
LinkedIn: https://linkedin.com/in/srivatsagorti
GitHub: https://github.com/srivatsa
Portfolio: https://srivatsa.dev
Company: www.buildsrivatsa.qzz.io
"""

url_pattern = r"https?://[\w./-]+"
urls = re.findall(url_pattern, url_text)
print("URLs found:")
for url in urls:
    print(f"  - {url}")

# Categorize
for url in urls:
    if "linkedin" in url.lower():
        print(f"  LinkedIn: {url}")
    elif "github" in url.lower():
        print(f"  GitHub: {url}")

URLs found:
  - https://linkedin.com/in/srivatsagorti
  - https://github.com/srivatsa
  - https://srivatsa.dev
  LinkedIn: https://linkedin.com/in/srivatsagorti
  GitHub: https://github.com/srivatsa


## 9. Date & Duration Extraction

Employment history is the heart of a resume, and date ranges are its structure. The pattern `([A-Z][a-z]+\s+\d{4})\s*-\s*([A-Z][a-z]+\s+\d{4}|Present)` captures start and end as two groups; the `|Present` alternative handles current roles.

**What the code does:**
- Month-year spans: `Jan 2020 → Present` and `June 2017 → August 2017` from the stored output.
- The looser year-only pattern `(\d{4})\s*-\s*(\d{4}|Present)` catches the bare-year lines: `2020 → Present`, `2018 → 2020`, `2016 → 2020` — the last from the education line.

**Why it matters:** tenure calculation (years per role) starts from these two capture groups — the raw material for experience scoring later in the project.

In [18]:
date_text = """
Work Experience
Software Engineer at Google | Jan 2020 - Present
Data Analyst at Amazon | 2018 - 2020
Intern at Microsoft | June 2017 - August 2017
Education: B.Tech CSE, 2016 - 2020
"""

# Date range pattern
date_pattern = r"([A-Z][a-z]+\s+\d{4})\s*-\s*([A-Z][a-z]+\s+\d{4}|Present)"
matches = re.findall(date_pattern, date_text)
for start, end in matches:
    print(f"  {start} → {end}")

# Year-only ranges
year_pattern = r"(\d{4})\s*-\s*(\d{4}|Present)"
year_matches = re.findall(year_pattern, date_text)
for start, end in year_matches:
    print(f"  {start} → {end}")

  Jan 2020 → Present
  June 2017 → August 2017
  2020 → Present
  2018 → 2020
  2016 → 2020


## 10. Skill Extraction

Skill extraction is dictionary matching done right: for each known skill, build `\b` + `re.escape(skill)` + `\b` and search with `re.IGNORECASE`. The `\b` word boundary stops `Python` from matching inside `Pythonic`; `re.escape` neutralizes regex metacharacters in skill names like `C++`.

**What the code does:**
- Scans a summary plus technical-skills block against 29 known skills and returns 12 sorted matches — the stored output lists them (`AWS`, `Deep Learning`, `TensorFlow`, ...).
- Case-folding via `re.IGNORECASE` lets one pattern cover `Python`, `python`, `PYTHON`.

**Why it matters:** this is the exact mechanism behind resume–JD keyword matching — a skill found in both documents is a match, and `\b` is what keeps the match honest.

In [19]:
# Known skills list (from ESCO/O*NET)
known_skills = [
    "Python", "NLP", "Machine Learning", "Deep Learning",
    "TensorFlow", "PyTorch", "SQL", "Spark", "Hadoop",
    "Docker", "Kubernetes", "AWS", "Azure", "GCP",
    "Java", "Scala", "C++", "JavaScript", "TypeScript",
    "React", "Angular", "Node.js", "Flask", "FastAPI",
    "Pandas", "NumPy", "scikit-learn", "Tableau", "Power BI"
]

resume_text = """
Professional Summary
Senior Data Scientist with 5+ years experience in Python, NLP,
and Machine Learning. Expert in TensorFlow and PyTorch for
deep learning applications. Proficient with AWS, Docker, and SQL.

Technical Skills
Languages: Python, Java, JavaScript
Frameworks: TensorFlow, PyTorch, Flask
Tools: Docker, AWS, Git
"""

def extract_skills_regex(text: str, skills_list: list) -> list:
    """Find known skills in text using regex."""
    found = set()
    text_lower = text.lower()
    for skill in skills_list:
        # Word boundary ensures we match 'Python' not 'Pythonic'
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, text, re.IGNORECASE):
            found.add(skill)
    return sorted(found)

skills_found = extract_skills_regex(resume_text, known_skills)
print("Skills extracted:")
for s in skills_found:
    print(f"  - {s}")
print(f"Total: {len(skills_found)}")

Skills extracted:
  - AWS
  - Deep Learning
  - Docker
  - Flask
  - Java
  - JavaScript
  - Machine Learning
  - NLP
  - PyTorch
  - Python
  - SQL
  - TensorFlow
Total: 12


## 11. Education Extraction

Degree extraction is harder than it looks: abbreviations vary (`B.Tech`, `BE`, `M.S.`, `PhD`), and a naive alternation matches *anywhere* — including inside unrelated words.

**What the code does:**
- `(B\.?Tech|M\.?S\.?|PhD|B\.?E\.?|M\.?Tech|MBA|B\.?Sc|M\.?Com)[^,]*` with `re.IGNORECASE` finds `B.Tech`, `M.S.`, `PhD` — and the stored output shows the failure mode: **`mba` appears twice**, matched inside *Mumbai* and *Bombay*.
- The institution pattern `(?:from|at)\s+([A-Z][A-Za-z\s.]+?)` finds **nothing** — the sample says "B.Tech *in* Computer Science, IIT Bombay", i.e. `in`, not `at`/`from`. Patterns are literal; the surface form must match.

**Why it matters:** two real lessons in one cell — anchor patterns to context (line start, degree keywords), and test against the actual text formats your data uses.

In [20]:
edu_text = """
Education:
- B.Tech in Computer Science, IIT Bombay, 2020
- M.S. in Data Science, Stanford University, 2022
- PhD in NLP, MIT (ongoing)
- Certifications: AWS Solutions Architect, Google Data Engineer
"""

degree_pattern = r"(B\.?Tech|M\.?S\.?|PhD|B\.?E\.?|M\.?Tech|MBA|B\.?Sc|M\.?Com)[^,]*"
degrees = re.findall(degree_pattern, edu_text, re.IGNORECASE)
print("Degrees found:")
for d in degrees:
    print(f"  {d.strip()}")

# University/institution extraction
uni_pattern = r"(?:from|at)\s+([A-Z][A-Za-z\s.]+?)(?=\s*,|\s+\d{4}|$)"
unis = re.findall(uni_pattern, edu_text)
print("\nInstitutions:")
for u in unis:
    print(f"  {u.strip()}")

Degrees found:
  B.Tech
  mba
  M.S.
  PhD

Institutions:


## 12. Location & Address Extraction

Location lines are usually structured enough for a city/state pattern: capitalized words, a comma, then either a 2-letter state code (`CA`) or another capitalized name (`India`).

**What the code does:**
- `([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*),\s*([A-Z]{2}|[A-Z][a-z]+)` handles multi-word cities via the inner `(?:\s+...)*` group.
- The stored output extracts `San Francisco, CA` and `Mumbai, India` — and skips `US ***` (work authorization) and `Yes` (relocation), since neither is a capitalized phrase before a comma.

**Why it matters:** location feeds relocation-eligibility and geographic matching in the ATS — cheap signal, easy to extract.

In [21]:
location_text = """
Location: San Francisco, CA
Current: Mumbai, India
Work Authorization: US Citizen
Willing to relocate: Yes
"""

# City, State pattern
location_pattern = r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*),\s*([A-Z]{2}|[A-Z][a-z]+)"
locations = re.findall(location_pattern, location_text)
print("Locations found:")
for city, state in locations:
    print(f"  {city}, {state}")

Locations found:
  San Francisco, CA
  Mumbai, India


## 13. Putting It All Together — Resume Extractor

This is the chapter's deliverable: `ExtractedResume`, a dataclass holding every field, and `extract_all()`, which runs all the section patterns over one text in a single pass. It is the direct ancestor of the Part II extraction pipeline.

**What the code does:**
- Name via the first-line heuristic; emails/phones/URLs via `list(set(re.findall(...)))` (dedupe included); skills via the `\b` + `re.escape` loop; degrees and locations via the section patterns.
- The stored output parses the sample completely: **Srivatsa Gorti**, one email, one phone, one LinkedIn URL, five skills.
- Education shows the section-11 leak again: `['B.Tech', 'mba', 'mba']` — `mba` from *Mumbai* and *Bombay*, duplicates included because this loop does not dedupe.

**Try it:** keep this extractor as your baseline — later chapters replace its heuristics (name, degrees) with statistical models, and this is what you diff against.

In [22]:
import re
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ExtractedResume:
    name: Optional[str] = None
    emails: List[str] = field(default_factory=list)
    phones: List[str] = field(default_factory=list)
    urls: List[str] = field(default_factory=list)
    skills: List[str] = field(default_factory=list)
    education: List[str] = field(default_factory=list)
    locations: List[str] = field(default_factory=list)

def extract_all(text: str, known_skills: list) -> ExtractedResume:
    res = ExtractedResume()

    # Name heuristic: first substantial line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines and len(lines[0].split()) in [2, 3]:
        res.name = lines[0]

    # Emails
    res.emails = list(set(re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text)))

    # Phones
    res.phones = list(set(re.findall(r"[+]?[\d\s()-]{7,}[\d]", text)))

    # URLs
    res.urls = list(set(re.findall(r"https?://[\w./-]+", text)))

    # Skills
    for skill in known_skills:
        if re.search(r"\b" + re.escape(skill) + r"\b", text, re.IGNORECASE):
            res.skills.append(skill)

    # Degrees
    degree_pat = r"(B\.?Tech|M\.?S\.?|PhD|B\.?E\.?|M\.?Tech|MBA|B\.?Sc)[^,]*"
    res.education = [d.strip() for d in re.findall(degree_pat, text, re.IGNORECASE)]

    # Locations
    loc_pat = r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*),\s*([A-Z]{2}|[A-Z][a-z]+)"
    res.locations = [f"{c}, {s}" for c, s in re.findall(loc_pat, text)]

    return res

# Test
sample = """Srivatsa Gorti
srivatsa@email.com | +91-9876543210
https://linkedin.com/in/srivatsa

Senior Data Scientist with expertise in Python, NLP,
and Machine Learning using TensorFlow and PyTorch.

Education: B.Tech in CSE, IIT Bombay, 2019
Location: Mumbai, India
"""

known = ["Python", "NLP", "Machine Learning", "TensorFlow", "PyTorch",
         "SQL", "Spark", "Docker", "AWS"]
result = extract_all(sample, known)
print(f"Name:      {result.name}")
print(f"Emails:    {result.emails}")
print(f"Phones:    {result.phones}")
print(f"URLs:      {result.urls}")
print(f"Skills:    {result.skills}")
print(f"Education: {result.education}")
print(f"Location:  {result.locations}")

Name:      Srivatsa Gorti
Emails:    ['srivatsa@email.com']
Phones:    ['+91-9876543210']
URLs:      ['https://linkedin.com/in/srivatsa']
Skills:    ['Python', 'NLP', 'Machine Learning', 'TensorFlow', 'PyTorch']
Education: ['B.Tech', 'mba', 'mba']
Location:  ['Python, NL', 'Mumbai, India']


## Regex Cheat Sheet

| Pattern | Matches | Example |
|---------|---------|---------|
| `\d` | Digit | `7`, `9` |
| `\w` | Word char | `a`, `B`, `_`, `9` |
| `\s` | Whitespace | space, tab, newline |
| `.` | Any char (except newline) | |
| `*` | 0 or more | `\w*` |
| `+` | 1 or more | `\d+` |
| `?` | 0 or 1 | `https?` |
| `{2,4}` | 2 to 4 times | `\d{2,4}` |
| `[abc]` | Any of a,b,c | |
| `[^abc]` | Not a,b,c | |
| `(x\|y)` | Either x or y | |
| `\b` | Word boundary | `\bPython\b` |
| `^` | Start of string | |
| `$` | End of string | |
| `(?P<n>...)` | Named group | `(?P<email>...)` |

## Summary

Today you learned:
- ✅ Regex fundamentals (raw strings, metacharacters, quantifiers)
- ✅ Character classes and groups
- ✅ Email, phone, URL extraction from resumes
- ✅ Date/duration parsing
- ✅ Skill extraction with word boundaries
- ✅ Education and location extraction
- ✅ A complete regex-based resume extractor

Regex will be your primary tool in Parts I & II of this project.

**How to use this sheet:** every pattern in this book is a combination of these pieces — class + quantifier + anchor + group — so this table is the reference for all of Part I. The summary below is the chapter's contract: regex is fast, deterministic, and explainable, which is why it stays the primary text tool for Parts I and II.

## Debugging Regex Like a Pro

Regex bugs are silent: a pattern that *almost* matches returns nothing, not an error. A short discipline fixes most of them.

- **Start minimal, grow slowly** — build `\d{4}` before `(\d{4})\s*-\s*(\d{4}|Present)`; each added piece is a testable hypothesis.
- **Test against the real text** — the section-11 institution pattern failed because the sample said `in`, not `at`; patterns are literal, so sample data must be representative.
- **Anchor unanchored alternations** — `(B\.?Tech|MBA|...)` matched *Mumbai*; use `^`, `\b`, or surrounding context when match location matters.
- **Prefer named groups and `re.VERBOSE`** for multi-part patterns — `(?P<field>...)` turns a match into a labeled record.
- **Escape user content** — `re.escape(skill)` before embedding a skill name in a pattern.

The section-11 and section-13 stored outputs above are worked examples of every rule in this list.

## Limits of Regex

Regex is literal and context-free: it matches *characters*, not *meaning*. It cannot tell `mba` inside *Mumbai* from an MBA degree, cannot know that "develop" and "development" share a root, and cannot handle paraphrase.

Every failure in this chapter is a regex limit: degree false positives inside city names, the missed `www.` URL, the institution pattern beaten by `in` instead of `at`, phone formats with dots. Each is a *surface-form* problem — same information, different string.

**Why it matters:** this is exactly why the next chapters exist — normalization (Ch. 06), lemmatization (Ch. 08), and statistical parsing (Ch. 10–12) handle the variation regex cannot. Regex owns the structured fields; NLP takes the ambiguous ones.

## Key Insight

**Regex is the fastest path from raw text to structured fields — and its limits define the NLP pipeline that follows.**

The extractor you built here — contact, URLs, dates, skills, degrees, locations — is a complete, explainable ATS field-extraction layer in a few dozen lines. Its failures are not bugs but boundaries: where surface forms vary, statistical NLP takes over. The patterns and the discipline (anchoring, escaping, testing on real text) carry into every later chapter.

Next, Ch. 04 — NLP Introduction — frames where regex ends and language models begin.